In [4]:
import sys
from pathlib import Path

# Aponta para a pasta 'src' dentro de 'rbg'
src_path = Path.cwd().parents[1] / "rbg" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Pacotes externos
import altair as alt
import numpy as np
import requests
import tarfile
import io
import pandas as pd

# Módulos da sua biblioteca 'rbg'
from rbg.Utilities import train_test_split, Accuracy, F1Score
from rbg.ModelSelection import StratifiedKFold, GridSearch
from rbg.Preprocessing import Standard
from rbg.Classification import KNN
from rbg.Unsupervised import PCA

In [5]:
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

CLASS_COLORS = {
    "DERMASON": "#0173B2",
    "SIRA":     "#DE8F05",
    "SEKER":    "#029E73",
    "HOROZ":    "#D55E00",
    "CALI":     "#CC78BC",
    "BARBUNYA": "#CA9161",
    "BOMBAY":   "#FBAFE4",
}

def stylize(chart):
    return (
        chart
        .configure_axis(labelFontSize=10, titleFontSize=11, gridColor="#E5E5E5")
        .configure_legend(symbolOpacity=1, labelFontSize=10, titleFontSize=11)
        .configure_title(fontSize=14)
    )

In [6]:
%pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

display(df[:5])
display(df['Class'].value_counts())

Note: you may need to restart the kernel to use updated packages.


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

# Processamento dos dados usando PCA e KNN
## Pré-processamento e Prevenção de Vazamento de Dados (Data Leakage)

- **Padronização das Variáveis** ($Z$-score)**:** Tanto o PCA quanto o KNN dependem diretamente da escala geométrica dos atributos numéricos. Sem a padronização, atributos com ampla variação escalar (como ```Area```, que pode chegar a mais de 1000) dominariam desproporcionalmente o cálculo das distâncias e a orientação dos componentes principais em relação a atributos delimitados (como ```Solidity```, que se situa no intervalo $(0, 1]$).

- **Isolamento Estrito dos Dados de Teste:** O dataset é dividido inicialmente em conjuntos de treino e teste pela função ```train_test_split```. O conjunto de teste é completamente reservado antes de qualquer cálculo estatístico.

- **Prevenção de Vazamento de Informação:** A média ($\mu$) e o desvio padrão ($\sigma$) das variáveis são calculados exclusivamente sobre o conjunto de treino via ```scaler.fit(X_train)```. As transformações (```scaler.transform()```) em ```X_train``` e ```X_test``` utilizam unicamente os parâmetros extraídos do treino, impedindo que a distribuição estatística do conjunto de teste contamine o aprendizado do modelo.

In [7]:
# Separação entre dados de treino e teste:
X = df.drop('Class', axis=1).values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle = True, random_state=42)

In [8]:
# Normalização dos dados:
standard_scaler = Standard()
standard_scaler.fit(X_train)
X_train = standard_scaler.transform(X_train)
# Para evitar vazamento de dados mas manter a padronização entre todos os dados,
# normalizamos os dados do X de teste também pela média e dp dos dados de treino:
X_test = standard_scaler.transform(X_test)

## Redução de Dimensionalidade com PCA (*Principal Component Analysis*)

Existem alguns motivos para aplicarmos o PCA antes do KNN, dentre eles:

- **Maldição da Dimensionalidade (*Curse of Dimensionality*):** Em espaços de alta dimensão, as distâncias entre pares de pontos tendem a se igualar, o que diminui a capacidade da métrica euclidiana usada no KNN. O PCA projeta os dados em um subespaço de menor dimensão, tornando as métricas de distância do KNN mais significativas e computacionalmente eficientes.

- **Eliminação de Colinearidade:** Atributos altamente correlacionados (como ```Area```, ```Perimeter```, ```MajorAxisLength``` e ```MinorAxisLength```) adicionam redundância. O PCA projeta os dados em eixos ortogonais, ou seja, pouco correlacionados, de forma que features "redundantes" acabam não sendo utilizadas.

- **Seleção por Variância Acumulada:** Em vez de escolher um número arbitrário de componentes, define-se uma meta de retenção da variância total de $95\%$. Essa escolha preserva a maior parte da informação estatística essencial do conjunto de dados, descartando apenas variações de baixo impacto e ruídos geométricos de alta dimensão.

- **Integridade do Pipeline de PCA:** A decomposição SVD e a matriz de autovetores são aprendidas apenas sobre o treino padronizado (```pca.fit(X_train_scaled)```). O conjunto de teste padronizado é projetado utilizando a mesma base ortogonal já calculada (```pca.transform(X_test_scaled)```).

In [9]:
pca = PCA(n_components=0.9) # variância acumulada desejada
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Dimensão original das features: {X_train.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Dimensão original das features: 16
Componentes mantidos pelo PCA: 4
Variância acumulada: 95.04%


## Classificação com KNN (K-Nearest Neighbors)

- **Mecanismo de Decisão:** O KNN atribui a classe de uma nova amostra com base na votação por maioria simples de seus $k$ vizinhos mais próximos no espaço projetado pelo PCA. A ideia de usar GridSearch para decidir qual o melhor hiperparâmetro $k$ é porque sabemos que ele não pode atingir um valor muito grande, pois um valor menor para esse hiperparâmetro faz com que o algoritmo capture melhor padrões locais, o que é essencial para a classificação de um dado novo. Portanto, decidir qual o $k$ que, nesse caso, aumenta nossa métrica é indispensável para fazermos previsões melhores.
- **Pipeline de Seleção de Hiperparâmetros (`GridSearch`):**
  - Validade da Validação Cruzada (`StratifiedKFold`): Aplicamos a validação cruzada sobre os dados de treino com menos features (`X_train_pca`), garantindo que a proporção das classes de feijões seja preservada em cada fold.
  - Uso de $k$ Ímpares: A busca no GridSearch limita a escolha do hiperparâmetro $k$ a valores ímpares, prevenindo cenários de empate na votação da vizinhança.
  - Métrica de Desempenho: Configuramos `greater_is_better=True` no GridSearch para otimizar métricas diretas de classificação, no nosso caso, a F1 Score, que foi utilizada justamente devido ao desbalanceamento de classes.


In [10]:
_, X_mini, _, y_mini = train_test_split(X_train_pca, y_train, test_size=1000)

valores_de_k = [3, 5, 7, 9]
grid = GridSearch(
    model=KNN,
    param_grid={'k': valores_de_k},
    metrics=F1Score,
    greater_is_better=True,
    cv=StratifiedKFold(k=5)
)

#grid.fit(X_train_pca, y_train)
grid.fit(X_mini, y_mini)
best_k = grid.best_params_['k']
best_knn = KNN(k=best_k)
best_knn.fit(X_train_pca, y_train)
print("Melhor hiperparâmetro encontrado:", best_k)
print(f"F1 Score médio na validação cruzada: {grid.best_score_ * 100:.2f}%")

Melhor hiperparâmetro encontrado: 7
F1 Score médio na validação cruzada: 87.02%


## Avaliação Final do Modelo:
O modelo KNN otimizado e reajustado no treino completo é testado uma única vez sobre `X_test_pca`, gerando métricas de generalização sem viés.

In [11]:
y_pred_test = best_knn.predict(X_test_pca)
test_acc = Accuracy(y_test, y_pred_test)
f1score = F1Score(y_test, y_pred_test)
print(f"F1 Score média nos conjuntos: {np.mean(f1score) * 100:.2f}%")
print(f"Acurácia final no conjunto de teste: {test_acc * 100:.2f}%")

F1 Score média nos conjuntos: 88.93%
Acurácia final no conjunto de teste: 88.51%


## Mais testes com o PCA

Para de fato enxergar como o KNN funciona, podemos usar o PCA para diminuir nossos dados a apenas dois features e plotar os dados em um espaço $\mathbb{R}^2$.

In [12]:
pca = PCA(n_components=2)
pca.fit(X_train)

X_train_pca2d = pca.transform(X_train)
X_test_pca2d = pca.transform(X_test)

print(f"Dimensão original das features: {X_train.shape[1]}")
print(f"Componentes mantidos pelo PCA: {pca.components.shape[0]}")
print(f"Variância acumulada: {np.sum(pca.explained_variance_ratio) * 100:.2f}%")

Dimensão original das features: 16
Componentes mantidos pelo PCA: 2
Variância acumulada: 81.96%


In [13]:
import altair as alt
import pandas as pd
import numpy as np

# montagem do dataframe com os dois principais componentes determinados pelo pca:
df_pca_plot = pd.DataFrame({"PC1": X_train_pca2d[:, 0],"PC2": X_train_pca2d[:, 1],"Class": y_train})
# escala de cores com dicionário CLASS_COLORS:
color_scale = alt.Scale(domain=list(CLASS_COLORS.keys()),range=list(CLASS_COLORS.values()))
# gráfico de dispersão com altair:
chart = (
    alt.Chart(df_pca_plot)
    .mark_circle(size=30, opacity=0.7)
    .encode(
        x=alt.X("PC1:Q", title="Componente Principal 1 (PC1)"),
        y=alt.Y("PC2:Q", title="Componente Principal 2 (PC2)"),
        color=alt.Color("Class:N", scale=color_scale, title="Classe"),
        tooltip=[
            "Class:N",
            alt.Tooltip("PC1:Q", format=".2f"),
            alt.Tooltip("PC2:Q", format=".2f")
        ]
    )
    .properties(
        title="Dispersão das Amostras com os Dois Principais Componentes",
        width=650,
        height=450
    )
)

# exibição do gráfico final:
chart_stylized = stylize(chart)
display(chart_stylized)

alt.Chart(...)

A projeção bidimensional via PCA justifica os limites de desempenho do classificador: observa-se uma acentuada sobreposição espacial entre as distribuições das classes no plano. Em particular, os agrupamentos de 'CALI' e 'BARBUNYA' apresentam forte interceptação de densidade, devido à semelhança morfológica entre elas, enquanto 'SIRA' posiciona-se em uma região central de fronteira ambígua, sobrepondo-se a múltiplas classes adjacentes. Considerando a compressão de 16 atributos para apenas 2 componentes principais e a inerente complexidade dessas fronteiras, alcançar métricas de Acurácia e F1-Score (Macro) na casa dos 90% no conjunto de teste evidencia um desempenho altamente sólido e robusto do KNN.